# JAX-CrossCat Paper Benchmarks (Colab)

Generates all benchmark numbers and figures for the arXiv paper.

**Runtime**: GPU (T4 or better) — **~30-45 min total**

| Section | Time est. | Output |
|---------|-----------|--------|
| 2. JIT kernel speedups | ~5 min | Table 2 numbers |
| 3. Synthetic recovery | ~15-20 min | Table 1 + 3 figures |
| 4. Scalability curves | ~10-15 min | Figure 7 (3 panels) |
| 5. Export | instant | tar.gz archive |

In [1]:
# Cell 1: GPU check
!nvidia-smi

Wed Apr  1 23:24:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: Install jaxcross (preserves Colab's JAX+CUDA stack)
import os

WORKDIR = "/content/jaxcross"
BRANCH = "feat/arxiv-paper"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

# --no-deps preserves Colab's pre-installed JAX+CUDA to avoid ptxas mismatch
%pip install -e . --no-deps -q
%pip install matplotlib scikit-learn -q

Branch 'feat/arxiv-paper' set up to track remote branch 'feat/arxiv-paper' from 'origin'.
Switched to a new branch 'feat/arxiv-paper'
From https://github.com/sambhal-labs/jaxcross
 * branch            feat/arxiv-paper -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jax-crosscat (pyproject.toml) ... done


In [3]:
# Cell 3: Verify installation + imports
import gc
import json
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import crosscat
from crosscat.diagnostics import adjusted_rand_index, column_partition_ari
from crosscat.gibbs import (
    gibbs_sweep,
    transition_column_hypers,
    transition_crp_alphas,
    transition_row_assignments,
)
from crosscat.inference import dependence_matrix
from crosscat.model import initialize, log_joint
from crosscat.packed import pack_state
from crosscat.packed.kernels import (
    packed_gibbs_sweep,
    packed_transition_column_hypers,
    packed_transition_crp_alphas,
    packed_transition_row_assignments,
)
from crosscat.packed.state import unpack_state
from crosscat.synthetic import generate_crosscat_data
from crosscat.types import ColumnType

matplotlib.use("Agg")

RESULTS_DIR = Path("benchmarks/results/paper")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"jaxcross version: {crosscat.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")
assert jax.default_backend() == "gpu", "GPU not available!"

jaxcross version: 0.10.1
JAX version: 0.7.2
Devices: [CudaDevice(id=0)]
Backend: gpu


---
## 2. JIT Kernel Speedup Table (Table 2)

Compares unpacked (Python loops) vs packed (JIT+GPU) per-kernel timing.
Data: 200 rows × 10 cols, all 5 column types.

In [4]:
def time_fn(fn, *args, n_runs=3, **kwargs):
    """Time a function (mean of n_runs), with warmup."""
    result = fn(*args, **kwargs)  # warmup
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        # Block until GPU computation completes
        if hasattr(result, "column_assignments"):
            result.column_assignments.block_until_ready()
        elif hasattr(result, "view_row_assignments"):
            result.view_row_assignments.block_until_ready()
        times.append(time.perf_counter() - t0)
    return sum(times) / len(times), result


# Generate benchmark data: 200 rows, 10 cols, all 5 types
print("Generating data (200 rows, 10 cols, 5 types)...")
col_types_bench = [
    ColumnType.CONTINUOUS,
    ColumnType.CONTINUOUS,
    ColumnType.CATEGORICAL,
    ColumnType.CATEGORICAL,
    ColumnType.BINARY,
    ColumnType.BINARY,
    ColumnType.ORDINAL,
    ColumnType.CYCLIC,
    ColumnType.CYCLIC,
    ColumnType.CONTINUOUS,
]
bench_result = generate_crosscat_data(
    jax.random.key(42),
    200,
    col_types_bench,
    n_views=3,
    n_clusters=3,
    cluster_separation=5.0,
)
bench_data = bench_result["data"]

# Initialize and warm up unpacked path (3 sweeps)
key = jax.random.key(0)
k1, k2 = jax.random.split(key)
state = initialize(k1, bench_data, col_types_bench)
print("Warming up unpacked path (3 sweeps)...")
state = gibbs_sweep(k2, state, bench_data, n_sweeps=3)
packed = pack_state(state)
print(f"State: {state.n_views} views, log_joint={float(log_joint(state, bench_data)):.1f}")

# Benchmark individual kernels
print("\nKernel benchmarks (mean of 3 runs):")
print("-" * 70)

kernel_results = {}
total_orig, total_packed = 0.0, 0.0

for name, orig_fn, packed_fn, needs_data in [
    ("row_assignments", transition_row_assignments, packed_transition_row_assignments, True),
    ("column_hypers", transition_column_hypers, packed_transition_column_hypers, True),
    ("crp_alphas", transition_crp_alphas, packed_transition_crp_alphas, False),
]:
    key = jax.random.key(123)
    if needs_data:
        t_o, _ = time_fn(orig_fn, key, state, bench_data)
        t_p, _ = time_fn(packed_fn, key, packed, bench_data)
    else:
        t_o, _ = time_fn(orig_fn, key, state)
        t_p, _ = time_fn(packed_fn, key, packed)
    speedup = t_o / max(t_p, 1e-9)
    print(f"  {name:25s}  orig: {t_o:.4f}s  packed: {t_p:.4f}s  {speedup:.1f}x")
    kernel_results[name] = {
        "unpacked_s": round(t_o, 4),
        "packed_s": round(t_p, 4),
        "speedup": round(speedup, 1),
    }
    total_orig += t_o
    total_packed += t_p

print("-" * 70)
total_speedup = total_orig / max(total_packed, 1e-9)
print(
    f"  {'TOTAL':25s}  orig: {total_orig:.4f}s  packed: {total_packed:.4f}s  {total_speedup:.1f}x"
)
kernel_results["total"] = {
    "unpacked_s": round(total_orig, 4),
    "packed_s": round(total_packed, 4),
    "speedup": round(total_speedup, 1),
}

# Full sweep benchmark (3 sweeps)
print("\nFull sweep benchmark (3 sweeps):")
key = jax.random.key(456)
t_orig_sweep, _ = time_fn(
    gibbs_sweep,
    key,
    state,
    bench_data,
    n_sweeps=3,
    kernels=("row_assignments", "column_hypers", "crp_alphas"),
)
t_packed_sweep, _ = time_fn(packed_gibbs_sweep, key, packed, bench_data, n_sweeps=3)
sweep_speedup = t_orig_sweep / max(t_packed_sweep, 1e-9)
print(
    f"  Unpacked: {t_orig_sweep:.4f}s  Packed: {t_packed_sweep:.4f}s  Speedup: {sweep_speedup:.1f}x"
)
kernel_results["full_sweep_3"] = {
    "unpacked_s": round(t_orig_sweep, 4),
    "packed_s": round(t_packed_sweep, 4),
    "speedup": round(sweep_speedup, 1),
}

with open(RESULTS_DIR / "kernel_speedups.json", "w") as f:
    json.dump(kernel_results, f, indent=2)
print(f"\nSaved to {RESULTS_DIR / 'kernel_speedups.json'}")

# Free memory
del state, packed, bench_data, bench_result
gc.collect()

Generating data (200 rows, 10 cols, 5 types)...
Warming up unpacked path (3 sweeps)...
State: 3 views, log_joint=-3399.5

Kernel benchmarks (mean of 3 runs):
----------------------------------------------------------------------
  row_assignments            orig: 260.5145s  packed: 0.2761s  943.5x
  column_hypers              orig: 71.4523s  packed: 0.0044s  16301.5x
  crp_alphas                 orig: 0.6102s  packed: 0.0011s  571.0x
----------------------------------------------------------------------
  TOTAL                      orig: 332.5771s  packed: 0.2816s  1181.2x

Full sweep benchmark (3 sweeps):
  Unpacked: 1082.6577s  Packed: 25.8979s  Speedup: 41.8x

Saved to benchmarks/results/paper/kernel_speedups.json


33

---
## 3. Synthetic Structure Recovery (Table 1, Figures 1-3)

200 rows, 8 continuous columns, 2 views × 3 clusters.
10 chains × 200 sweeps (batched in chunks of 50 for throughput).

In [5]:
# ---- Config ----
N_CHAINS = 10
N_SWEEPS = 200
DIAG_INTERVAL = 50  # unpack for diagnostics every N sweeps
SEED = 42


def gen_synthetic(rng_key, n_rows=200):
    """Generate 2-view, 3-cluster synthetic data."""
    k1, k2, k3, k4 = jax.random.split(rng_key, 4)
    ca_v0 = jax.random.categorical(k1, jnp.log(jnp.array([1 / 3, 1 / 3, 1 / 3])), shape=(n_rows,))
    data_v0 = (
        jnp.array([[-3.0] * 4, [0.0] * 4, [3.0] * 4])[ca_v0]
        + jax.random.normal(k2, (n_rows, 4)) * 0.5
    )
    ca_v1 = jax.random.categorical(k3, jnp.log(jnp.array([1 / 3, 1 / 3, 1 / 3])), shape=(n_rows,))
    data_v1 = (
        jnp.array([[-4.0] * 4, [0.0] * 4, [4.0] * 4])[ca_v1]
        + jax.random.normal(k4, (n_rows, 4)) * 0.5
    )
    return (
        jnp.concatenate([data_v0, data_v1], axis=1),
        [ColumnType.CONTINUOUS] * 8,
        jnp.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=jnp.int32),
        [ca_v0, ca_v1],
    )


synth_data, synth_col_types, true_col, true_rows = gen_synthetic(jax.random.key(SEED))
print(f"Synthetic data: {synth_data.shape}")
print(f"Config: {N_CHAINS} chains x {N_SWEEPS} sweeps (diag every {DIAG_INTERVAL})")

rng_key = jax.random.key(SEED + 1)
init_keys = jax.random.split(rng_key, N_CHAINS)

states = []
sweep_metrics = []  # per-chain, per-checkpoint
t_total = time.time()

for chain_idx in range(N_CHAINS):
    print(f"\n--- Chain {chain_idx + 1}/{N_CHAINS} ---")
    k_i, k_sweep = jax.random.split(init_keys[chain_idx])
    state = initialize(k_i, synth_data, synth_col_types)
    packed = pack_state(state)
    print(f"  Init: {state.n_views} views")
    del state  # free CPU memory

    chain_ckpts = []
    t0 = time.time()

    # Run in batches of DIAG_INTERVAL sweeps for max GPU throughput
    sweep = 0
    while sweep < N_SWEEPS:
        batch = min(DIAG_INTERVAL, N_SWEEPS - sweep)
        k_sweep, subkey = jax.random.split(k_sweep)
        packed = packed_gibbs_sweep(subkey, packed, synth_data, n_sweeps=batch)
        sweep += batch

        # Diagnostics — unpack temporarily, extract metrics, then free
        state_tmp = unpack_state(packed, synth_col_types, data=synth_data)
        col_ari = float(column_partition_ari(state_tmp, true_col))
        n_views = state_tmp.n_views
        elapsed = time.time() - t0
        print(
            f"  Sweep {sweep:4d}/{N_SWEEPS}: views={n_views}, col_ARI={col_ari:.3f}, {elapsed:.0f}s"
        )
        chain_ckpts.append({"sweep": sweep, "n_views": n_views, "col_ari": col_ari})
        del state_tmp

    # Final unpack
    final_state = unpack_state(packed, synth_col_types, data=synth_data)
    elapsed = time.time() - t0
    print(f"  Done: {elapsed:.1f}s ({elapsed / N_SWEEPS:.3f}s/sweep)")

    states.append(final_state)
    sweep_metrics.append(chain_ckpts)
    del packed
    gc.collect()

print(f"\nTotal inference time: {time.time() - t_total:.1f}s")

Synthetic data: (200, 8)
Config: 10 chains x 200 sweeps (diag every 50)

--- Chain 1/10 ---
  Init: 5 views
  Sweep   50/200: views=1, col_ARI=-0.000, 47s
  Sweep  100/200: views=1, col_ARI=-0.000, 93s
  Sweep  150/200: views=1, col_ARI=-0.000, 135s
  Sweep  200/200: views=1, col_ARI=-0.000, 178s
  Done: 177.8s (0.889s/sweep)

--- Chain 2/10 ---
  Init: 2 views
  Sweep   50/200: views=1, col_ARI=-0.000, 42s
  Sweep  100/200: views=1, col_ARI=-0.000, 84s
  Sweep  150/200: views=1, col_ARI=-0.000, 127s
  Sweep  200/200: views=1, col_ARI=-0.000, 169s
  Done: 169.3s (0.846s/sweep)

--- Chain 3/10 ---
  Init: 4 views
  Sweep   50/200: views=2, col_ARI=1.000, 42s
  Sweep  100/200: views=2, col_ARI=1.000, 86s
  Sweep  150/200: views=2, col_ARI=1.000, 128s
  Sweep  200/200: views=2, col_ARI=1.000, 171s
  Done: 170.8s (0.854s/sweep)

--- Chain 4/10 ---
  Init: 3 views
  Sweep   50/200: views=2, col_ARI=1.000, 43s
  Sweep  100/200: views=2, col_ARI=1.000, 85s
  Sweep  150/200: views=2, col_ARI=1

In [6]:
# ---- Evaluate recovery metrics ----
print("=" * 60)
print("SYNTHETIC RECOVERY METRICS")
print("=" * 60)


def best_view_match(state, true_rows):
    return [
        max(float(adjusted_rand_index(tr, v.row_assignments)) for v in state.views)
        for tr in true_rows
    ]


col_aris = [float(column_partition_ari(s, true_col)) for s in states]
row_matches = [best_view_match(s, true_rows) for s in states]
row_v0 = [m[0] for m in row_matches]
row_v1 = [m[1] for m in row_matches]

good_chains = [i for i, s in enumerate(states) if s.n_views >= 2]
print(f"\nChains with 2+ views: {len(good_chains)}/{N_CHAINS}")
print(f"Per-chain views: {[s.n_views for s in states]}")
print(f"Per-chain col_ARI: {[round(a, 3) for a in col_aris]}")

z = dependence_matrix(states)
within = float((z[:4, :4].sum() + z[4:, 4:].sum() - 8.0) / (2 * (4 * 3)))
between = float(z[:4, 4:].mean())

best_idx = max(range(len(states)), key=lambda i: col_aris[i])
print(f"\nBest chain: {best_idx} (col_ARI={col_aris[best_idx]:.3f})")

synth_results = {
    "n_chains": N_CHAINS,
    "n_sweeps": N_SWEEPS,
    "chains_with_2plus_views": len(good_chains),
    "col_ari_mean": round(sum(col_aris) / len(col_aris), 4),
    "col_ari_best": round(max(col_aris), 4),
    "col_ari_per_chain": [round(a, 4) for a in col_aris],
    "row_ari_v0_mean": round(sum(row_v0) / len(row_v0), 4),
    "row_ari_v1_mean": round(sum(row_v1) / len(row_v1), 4),
    "within_view_dep": round(within, 4),
    "between_view_dep": round(between, 4),
    "views_per_chain": [s.n_views for s in states],
}

if good_chains:
    good_col = [col_aris[i] for i in good_chains]
    good_v0 = [row_v0[i] for i in good_chains]
    good_v1 = [row_v1[i] for i in good_chains]
    synth_results["good_chain_col_ari_mean"] = round(sum(good_col) / len(good_col), 4)
    synth_results["good_chain_row_ari_v0_mean"] = round(sum(good_v0) / len(good_v0), 4)
    synth_results["good_chain_row_ari_v1_mean"] = round(sum(good_v1) / len(good_v1), 4)
    print(f"  Good-chain col_ARI mean: {synth_results['good_chain_col_ari_mean']:.3f}")
    print(f"  Good-chain row_ARI v0:   {synth_results['good_chain_row_ari_v0_mean']:.3f}")
    print(f"  Good-chain row_ARI v1:   {synth_results['good_chain_row_ari_v1_mean']:.3f}")

print(f"\nZ-matrix within-view dep:  {within:.3f}")
print(f"Z-matrix between-view dep: {between:.3f}")

checks = [
    ("Col ARI (best) >= 0.80", synth_results["col_ari_best"] >= 0.80),
    ("Within-view dep >= 0.80", within >= 0.80),
    ("Between-view dep <= 0.20", between <= 0.20),
]
print()
for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

with open(RESULTS_DIR / "synthetic_results.json", "w") as f:
    json.dump(synth_results, f, indent=2)
print(f"\nSaved to {RESULTS_DIR / 'synthetic_results.json'}")

SYNTHETIC RECOVERY METRICS

Chains with 2+ views: 4/10
Per-chain views: [1, 1, 2, 2, 1, 1, 2, 2, 1, 1]
Per-chain col_ARI: [-0.0, -0.0, 1.0, 1.0, -0.0, -0.0, 1.0, 1.0, -0.0, -0.0]

Best chain: 2 (col_ARI=1.000)
  Good-chain col_ARI mean: 1.000
  Good-chain row_ARI v0:   1.000
  Good-chain row_ARI v1:   0.879

Z-matrix within-view dep:  1.000
Z-matrix between-view dep: 0.600

  [PASS] Col ARI (best) >= 0.80
  [PASS] Within-view dep >= 0.80
  [FAIL] Between-view dep <= 0.20

Saved to benchmarks/results/paper/synthetic_results.json


In [7]:
# ---- Synthetic Figures ----

# Figure 1: Convergence (all chains, good chains highlighted)
fig, ax = plt.subplots(figsize=(8, 4))
for i, ckpts in enumerate(sweep_metrics):
    sweeps = [c["sweep"] for c in ckpts]
    aris = [c["col_ari"] for c in ckpts]
    alpha = 1.0 if i in good_chains else 0.3
    lw = 2.0 if i in good_chains else 1.0
    ax.plot(
        sweeps,
        aris,
        "o-",
        alpha=alpha,
        linewidth=lw,
        label=f"Chain {i} ({states[i].n_views}v)" if i < 6 else None,
    )
ax.set_xlabel("Gibbs sweep", fontsize=11)
ax.set_ylabel("Column partition ARI", fontsize=11)
ax.set_title("Synthetic Recovery: Convergence", fontsize=12)
ax.set_ylim(-0.1, 1.1)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "synthetic_convergence.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: Z-matrix
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(np.array(z), cmap="Greens", vmin=0, vmax=1)
ax.set_title("Dependence Probability Matrix (Z-matrix)", fontsize=12)
col_labels = [f"V0-C{i}" for i in range(4)] + [f"V1-C{i}" for i in range(4)]
ax.set_xticks(range(8))
ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(8))
ax.set_yticklabels(col_labels, fontsize=9)
plt.colorbar(im, ax=ax, label="P(dependent)")
plt.tight_layout()
fig.savefig(RESULTS_DIR / "synthetic_z_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure 3: Cluster recovery scatter (best chain)
best_state = states[best_idx]
data_np = np.array(synth_data)
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]

# Find best-matching inferred views
view_match = []
for tr in true_rows:
    best_ari, best_vi = -1, 0
    for vi, v in enumerate(best_state.views):
        a = float(adjusted_rand_index(tr, v.row_assignments))
        if a > best_ari:
            best_ari, best_vi = a, vi
    view_match.append(best_vi)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for v, (cx, cy) in enumerate([(0, 1), (4, 5)]):
    true_a = np.array(true_rows[v])
    inf_a = np.array(best_state.views[view_match[v]].row_assignments)
    for cid in range(int(true_a.max()) + 1):
        m = true_a == cid
        axes[0, v].scatter(data_np[m, cx], data_np[m, cy], c=colors[cid % 5], s=15, alpha=0.6)
    for cid in range(int(inf_a.max()) + 1):
        m = inf_a == cid
        axes[1, v].scatter(data_np[m, cx], data_np[m, cy], c=colors[cid % 5], s=15, alpha=0.6)
    axes[0, v].set_title(f"True — View {v}", fontsize=11)
    axes[1, v].set_title(f"Inferred — View {v}", fontsize=11)
    for row in range(2):
        axes[row, v].set_xlabel(f"Column {cx}", fontsize=9)
        axes[row, v].set_ylabel(f"Column {cy}", fontsize=9)

fig.suptitle(
    f"Cluster Recovery (Best Chain {best_idx}, ARI={col_aris[best_idx]:.3f})", fontsize=13
)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "synthetic_cluster_recovery.png", dpi=300, bbox_inches="tight")
plt.show()

print("Synthetic figures saved.")

# Free synthetic state memory
del states, synth_data, best_state
gc.collect()

Synthetic figures saved.


39

---
## 4. Scalability Curves (Figure 7)

Per-sweep wall-clock time vs rows and columns.
Each point: compile once, then measure mean of 5 sweeps.

In [8]:
def make_data(key, n_rows, n_cols):
    """Generate random mixed-type data."""
    type_cycle = [
        ColumnType.CONTINUOUS,
        ColumnType.BINARY,
        ColumnType.CATEGORICAL,
        ColumnType.CONTINUOUS,
        ColumnType.BINARY,
    ]
    col_types = [type_cycle[i % len(type_cycle)] for i in range(n_cols)]
    parts = []
    for j in range(n_cols):
        kj = jax.random.fold_in(key, j)
        ct = col_types[j]
        if ct == ColumnType.CONTINUOUS:
            parts.append(jax.random.normal(kj, (n_rows,)) * 3.0)
        elif ct == ColumnType.BINARY:
            parts.append(jax.random.bernoulli(kj, 0.5, (n_rows,)).astype(jnp.float32))
        elif ct == ColumnType.CATEGORICAL:
            parts.append(jax.random.randint(kj, (n_rows,), 0, 5).astype(jnp.float32))
        else:
            parts.append(jax.random.normal(kj, (n_rows,)))
    return jnp.stack(parts, axis=1), col_types


def time_sweep_fn(key, data, col_types, n_sweeps=5):
    """Compile once, then time n_sweeps. Returns (compile_time, per_sweep_time)."""
    k1, k2, k3 = jax.random.split(key, 3)
    st = initialize(k1, data, col_types)
    pk = pack_state(st)
    del st
    # Compile
    t0 = time.perf_counter()
    pk = packed_gibbs_sweep(k2, pk, data, n_sweeps=1)
    pk.column_assignments.block_until_ready()
    compile_t = time.perf_counter() - t0
    # Timed (cached compilation)
    t0 = time.perf_counter()
    pk = packed_gibbs_sweep(k3, pk, data, n_sweeps=n_sweeps)
    pk.column_assignments.block_until_ready()
    per_sweep = (time.perf_counter() - t0) / n_sweeps
    del pk
    gc.collect()
    return compile_t, per_sweep


# ---- Scalability vs rows (fixed 10 cols) ----
row_counts = [50, 100, 200, 500, 1000, 2000]
rows_results = []
print("=== Scalability vs N_rows (N_cols=10) ===")
for nr in row_counts:
    k = jax.random.fold_in(jax.random.key(42), nr)
    d, ct = make_data(k, nr, 10)
    comp, sweep = time_sweep_fn(jax.random.fold_in(k, 999), d, ct)
    print(f"  N_rows={nr:5d}: compile={comp:.1f}s, sweep={sweep:.4f}s")
    rows_results.append(
        {"n_rows": nr, "n_cols": 10, "compile_time": round(comp, 2), "per_sweep": round(sweep, 4)}
    )
    del d
    gc.collect()

# ---- Scalability vs cols (fixed 200 rows) ----
col_counts = [5, 10, 20, 50, 100]
cols_results = []
print("\n=== Scalability vs N_cols (N_rows=200) ===")
for nc in col_counts:
    k = jax.random.fold_in(jax.random.key(42), nc + 10000)
    d, ct = make_data(k, 200, nc)
    comp, sweep = time_sweep_fn(jax.random.fold_in(k, 999), d, ct)
    print(f"  N_cols={nc:5d}: compile={comp:.1f}s, sweep={sweep:.4f}s")
    cols_results.append(
        {"n_rows": 200, "n_cols": nc, "compile_time": round(comp, 2), "per_sweep": round(sweep, 4)}
    )
    del d
    gc.collect()

scale_results = {
    "vs_rows": rows_results,
    "vs_cols": cols_results,
    "device": str(jax.devices()[0]),
}
with open(RESULTS_DIR / "scalability_results.json", "w") as f:
    json.dump(scale_results, f, indent=2)
print(f"\nSaved to {RESULTS_DIR / 'scalability_results.json'}")

=== Scalability vs N_rows (N_cols=10) ===
  N_rows=   50: compile=25.8s, sweep=5.2403s
  N_rows=  100: compile=26.0s, sweep=5.3065s
  N_rows=  200: compile=25.2s, sweep=5.5305s
  N_rows=  500: compile=26.9s, sweep=6.0536s
  N_rows= 1000: compile=28.4s, sweep=6.6896s
  N_rows= 2000: compile=29.5s, sweep=8.2617s

=== Scalability vs N_cols (N_rows=200) ===
  N_cols=    5: compile=29.2s, sweep=6.1715s
  N_cols=   10: compile=26.6s, sweep=5.5445s
  N_cols=   20: compile=26.8s, sweep=5.5138s
  N_cols=   50: compile=27.3s, sweep=5.7170s
  N_cols=  100: compile=27.6s, sweep=5.9602s

Saved to benchmarks/results/paper/scalability_results.json


In [9]:
# ---- Scalability Plots ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (a) vs rows
ax = axes[0]
xs = [r["n_rows"] for r in rows_results]
ys = [r["per_sweep"] for r in rows_results]
ax.plot(xs, ys, "o-", color="#2196F3", linewidth=2, markersize=6)
ax.set_xlabel("Number of rows", fontsize=11)
ax.set_ylabel("Time per sweep (s)", fontsize=11)
ax.set_title("(a) Scaling with rows (10 cols)", fontsize=12)
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (b) vs cols
ax = axes[1]
xs = [r["n_cols"] for r in cols_results]
ys = [r["per_sweep"] for r in cols_results]
ax.plot(xs, ys, "s-", color="#4CAF50", linewidth=2, markersize=6)
ax.set_xlabel("Number of columns", fontsize=11)
ax.set_ylabel("Time per sweep (s)", fontsize=11)
ax.set_title("(b) Scaling with columns (200 rows)", fontsize=12)
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# (c) Compile time
ax = axes[2]
sr = [r["n_rows"] * r["n_cols"] for r in rows_results]
cr = [r["compile_time"] for r in rows_results]
sc = [r["n_rows"] * r["n_cols"] for r in cols_results]
cc = [r["compile_time"] for r in cols_results]
ax.scatter(sr, cr, marker="o", color="#2196F3", label="Vary rows", s=50, zorder=5)
ax.scatter(sc, cc, marker="s", color="#4CAF50", label="Vary cols", s=50, zorder=5)
ax.set_xlabel("Table size (rows × cols)", fontsize=11)
ax.set_ylabel("JIT compile time (s)", fontsize=11)
ax.set_title("(c) Compilation overhead", fontsize=12)
ax.set_xscale("log")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "scalability.png", dpi=300, bbox_inches="tight")
plt.show()
print("Scalability plots saved.")

Scalability plots saved.


---
## 5. Export All Results

In [10]:
import subprocess

print("=" * 60)
print("PAPER BENCHMARK RESULTS SUMMARY")
print("=" * 60)
print(f"\nDevice: {jax.devices()[0]}")
print(f"JAX: {jax.__version__}, jaxcross: {crosscat.__version__}")

print("\n--- Kernel Speedups (Table 2) ---")
for k, v in kernel_results.items():
    if isinstance(v, dict):
        print(
            f"  {k:25s}  unpacked={v['unpacked_s']:.4f}s  packed={v['packed_s']:.4f}s  {v['speedup']:.1f}x"
        )

print("\n--- Synthetic Recovery (Table 1) ---")
for k, v in synth_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    elif isinstance(v, int):
        print(f"  {k}: {v}")

print("\n--- Scalability ---")
for r in rows_results:
    print(
        f"  {r['n_rows']:5d} rows x {r['n_cols']:3d} cols: {r['per_sweep']:.4f} s/sweep (compile: {r['compile_time']:.1f}s)"
    )
for r in cols_results:
    print(
        f"  {r['n_rows']:5d} rows x {r['n_cols']:3d} cols: {r['per_sweep']:.4f} s/sweep (compile: {r['compile_time']:.1f}s)"
    )

# Archive all results
archive = "/content/paper_benchmark_results.tar.gz"
subprocess.run(
    ["tar", "czf", archive, "-C", str(RESULTS_DIR.parent), RESULTS_DIR.name],
    check=True,
)
print(f"\nArchive: {archive}")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

print("\nDownload from Colab Files panel (left sidebar).")

PAPER BENCHMARK RESULTS SUMMARY

Device: cuda:0
JAX: 0.7.2, jaxcross: 0.10.1

--- Kernel Speedups (Table 2) ---
  row_assignments            unpacked=260.5145s  packed=0.2761s  943.5x
  column_hypers              unpacked=71.4523s  packed=0.0044s  16301.5x
  crp_alphas                 unpacked=0.6102s  packed=0.0011s  571.0x
  total                      unpacked=332.5771s  packed=0.2816s  1181.2x
  full_sweep_3               unpacked=1082.6577s  packed=25.8979s  41.8x

--- Synthetic Recovery (Table 1) ---
  n_chains: 10
  n_sweeps: 200
  chains_with_2plus_views: 4
  col_ari_mean: 0.4000
  col_ari_best: 1.0000
  row_ari_v0_mean: 0.7046
  row_ari_v1_mean: 0.5763
  within_view_dep: 1.0000
  between_view_dep: 0.6000
  good_chain_col_ari_mean: 1.0000
  good_chain_row_ari_v0_mean: 1.0000
  good_chain_row_ari_v1_mean: 0.8793

--- Scalability ---
     50 rows x  10 cols: 5.2403 s/sweep (compile: 25.8s)
    100 rows x  10 cols: 5.3065 s/sweep (compile: 26.0s)
    200 rows x  10 cols: 5.5305 s/s